# Métricas de retrieval para RAG

## Objetivos

Evaluaremos el *retriever* antes de culpar a la respuesta final. Las consultas se ejecutan sobre las páginas extraídas de los PDFs reales de manuales de lavadora del curso, no sobre un ranking escrito a mano. Calcularemos `recall@k`, `precision@k` y MRR desde cero. Al finalizar sabrás distinguir cobertura, ruido y posición de la evidencia, además de decidir qué componente del pipeline ajustar.

**Caso guiado:** `R02`. La evidencia de instalación sí fue recuperada, pero quedó en la posición 2.

## 1. Carga de los datos y del eval set

Esta primera sección todavía **no calcula métricas**. Carga las preguntas, referencias y etiquetas doradas del eval set; después ejecuta el retriever TF-IDF sobre páginas extraídas con PyMuPDF desde los PDFs originales para preparar los rankings que se evaluarán más adelante.

Las preguntas que sí tienen evidencia se usarán en la siguiente sección para calcular `recall@k`, `precision@k` y MRR. Las preguntas trampa, sin evidencia en el corpus, se evalúan por **abstención**: su objetivo es que el sistema no invente una respuesta.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
from metrics_course.datasets import load_rag_cases
from metrics_course.real_retrieval import verify_pdf_provenance

cases = load_rag_cases()
case_catalog = pd.DataFrame([
    {
        'case_id': case['id'],
        'category': case['category'],
        'question': case['question'],
        'top_3_pages_returned': len(case['contexts']),
        'gold_pages_found_in_top_3': sum(context['relevant'] for context in case['contexts']),
    }
    for case in cases
])
print('Carga completada — Salida real del retriever: todavía no son métricas.')
display(case_catalog)

provenance = pd.DataFrame(verify_pdf_provenance(cases))
print('Validación del eval set — Auditoría de fuentes doradas: tampoco es una métrica.')
display(provenance[['source', 'exists', 'hash_matches']])

guided_case_id = 'R02'
guided_case = next(case for case in cases if case['id'] == guided_case_id)
print(f"Caso guiado: {guided_case_id} — {guided_case['question']}")


### Interpretación inmediata: auditoría antes de evaluar

La primera tabla no muestra `recall`, `precision` ni MRR. Sólo responde: “el retriever devolvió tres páginas; ¿cuántas coinciden con las páginas doradas?”. Por ejemplo, `R02` encuentra una página dorada en el top-3; `R01` no encuentra la suya. La segunda tabla confirma que los PDFs usados por las etiquetas realmente existen y no cambiaron.

**Qué sigue:** la sección 2 toma exactamente esta salida del retriever y calcula las métricas. El ranking es real; la etiqueta dorada es la referencia humana para decidir si acertó.

## 2. Recall@k — ¿la evidencia correcta llegó al contexto?

`recall@k = relevantes recuperados en los primeros k / relevantes existentes`.

Pregunta que responde: **¿trajimos toda la evidencia necesaria dentro de los primeros `k` resultados?** Un valor de `1.000` significa que sí llegó toda la evidencia marcada como necesaria; `0.000` significa que no llegó ninguna.

Subir `k` puede aumentar recall porque se permiten más páginas en el contexto. Las secciones siguientes mostrarán el costo de esa decisión: también puede entrar ruido.

In [ ]:
from metrics_course.retrieval import precision_at_k, recall_at_k, reciprocal_rank

retrieval_cases = [case for case in cases if case['category'] != 'trap']
ks = [1, 2, 3]
rows = []
rankings = []

for case in retrieval_cases:
    ranked_contexts = sorted(case['contexts'], key=lambda context: context['rank'])
    relevance = [context['relevant'] for context in ranked_contexts]
    rankings.append(relevance)
    row = {'case_id': case['id'], 'reciprocal_rank': reciprocal_rank(relevance)}
    for k in ks:
        row[f'recall@{k}'] = recall_at_k(relevance, k)
        row[f'precision@{k}'] = precision_at_k(relevance, k)
    rows.append(row)

per_case_metrics = pd.DataFrame(rows)
recall_columns = ['case_id', 'recall@1', 'recall@2', 'recall@3']
print('Evaluación de Recall@k — ranking real comparado contra las páginas doradas.')
display(per_case_metrics[recall_columns].style.format({column: '{:.3f}' for column in recall_columns if column != 'case_id'}))

guided_metrics = per_case_metrics.loc[per_case_metrics.case_id == guided_case_id].iloc[0]
print(
    f"{guided_case_id}: recall@1={guided_metrics['recall@1']:.3f}, "
    f"recall@2={guided_metrics['recall@2']:.3f}, "
    f"recall@3={guided_metrics['recall@3']:.3f}"
)


### Interpretación de Recall@k: R02 — la evidencia llega al usar `k=2`

En `R02`, `recall@1 = 0.000`: el primer resultado no contiene la evidencia de instalación. Al pasar a `k=2`, `recall@2 = 1.000`: ya entró toda la evidencia relevante. Este número sólo responde si llegó la evidencia; todavía no dice cuánta basura entró ni en qué posición exacta apareció.

**Veredicto:** la cobertura es buena si se permite `k=2`. En la siguiente sección revisaremos precision@k para saber cuánto ruido se agregó al aumentar `k`.

## 3. Precision@k — ¿cuánto ruido entró al contexto?

`precision@k = relevantes recuperados en los primeros k / resultados mostrados en los primeros k`.

Pregunta que responde: **de las páginas que se entregaron al LLM, ¿qué proporción realmente es útil?** Un valor de `1.000` significa que no entró ruido; un valor de `0.000` significa que ninguna página mostrada ayuda a responder.

In [ ]:
precision_columns = ['case_id', 'precision@1', 'precision@2', 'precision@3']
print('Evaluación de Precision@k — fracción de páginas útiles dentro del contexto entregado.')
display(per_case_metrics[precision_columns].style.format({
    column: '{:.3f}' for column in precision_columns if column != 'case_id'
}))

print(
    f"{guided_case_id}: precision@1={guided_metrics['precision@1']:.3f}, "
    f"precision@2={guided_metrics['precision@2']:.3f}, "
    f"precision@3={guided_metrics['precision@3']:.3f}"
)


### Interpretación de Precision@k: R02 — hay evidencia, pero también ruido

En `R02`, `precision@1 = 0.000` porque la primera página no ayuda con la instalación. `precision@2 = 0.500`: al incluir dos páginas, una es útil y una es ruido. `precision@3 = 0.333`: al agregar una tercera página irrelevante, la proporción útil vuelve a bajar.

**Veredicto:** aumentar `k` arregla recall, pero no arregla el orden. **Qué hacer:** probar *re-ranking* o mejorar la consulta para que la página 5 suba a rango 1.

## 4. MRR — ¿qué tan arriba apareció la primera evidencia útil?

Primero calculamos `reciprocal rank = 1 / posición del primer resultado relevante` para cada pregunta. Después calculamos `MRR` como el promedio de esos valores.

Pregunta que responde: **¿la primera evidencia útil quedó arriba de la lista?** MRR vale `1.000` si todas las preguntas aciertan en rango 1; baja cuando la evidencia aparece más tarde o no aparece.

In [ ]:
from metrics_course.retrieval import mean_reciprocal_rank

mrr_by_case = per_case_metrics[['case_id', 'reciprocal_rank']].rename(
    columns={'reciprocal_rank': 'reciprocal_rank = 1 / first_relevant_rank'}
)
mrr = mean_reciprocal_rank(rankings)

print('Evaluación de MRR — posición de la primera página relevante.')
display(mrr_by_case.style.format({'reciprocal_rank = 1 / first_relevant_rank': '{:.3f}'}))
print(f'MRR del conjunto (sin preguntas trampa): {mrr:.3f}')
guided_first_rank = int(1 / guided_metrics['reciprocal_rank'])
print(
    f"{guided_case_id}: primera evidencia en rango {guided_first_rank} → "
    f"reciprocal rank = 1 / {guided_first_rank} = {guided_metrics['reciprocal_rank']:.3f}"
)


### Interpretación de MRR: R02 — la evidencia llega tarde

`R02` tiene *reciprocal rank* de `0.500` porque la primera página relevante aparece en rango 2: `1 / 2 = 0.500`. Esta métrica no cuenta todas las páginas útiles; se enfoca únicamente en qué tan pronto aparece la primera.

**Veredicto:** MRR es la métrica principal para revisar orden y *re-ranking*. Si recall es alto pero MRR bajo, la evidencia sí está, pero el usuario y el LLM deben atravesar ruido antes de verla.

## 5. Ver la lista que produjo los números

Los promedios no sustituyen la inspección de un caso. Esta tabla muestra exactamente qué fragmento ocupa cada rango en `R02`.

In [ ]:
guided_contexts = pd.DataFrame(
    sorted(guided_case['contexts'], key=lambda context: context['rank'])
)
guided_contexts['text_preview'] = (
    guided_contexts['text'].str.replace('\n', ' ', regex=False).str.slice(0, 260) + '…'
)

def highlight_relevant(row):
    if row['relevant']:
        return ['background-color: #1F5A42; color: #FFFFFF; font-weight: 600'] * len(row)
    return [''] * len(row)

display(
    guided_contexts[[
        'rank', 'id', 'relevant', 'source', 'page', 'section',
        'retrieval_score', 'text_preview'
    ]].style
    .format({'retrieval_score': '{:.3f}'})
    .apply(highlight_relevant, axis=1)
)

first_relevant_rank = int(guided_contexts.loc[guided_contexts.relevant, 'rank'].iloc[0])
print(f'La primera evidencia relevante está en rank {first_relevant_rank}.')


### Interpretación inmediata: por qué el ranking importa

La tabla muestra una vista previa del texto extraído literalmente del PDF, junto con archivo, página, sección y puntuación del retriever. La fila verde oscuro es la evidencia marcada como relevante; el texto blanco mantiene contraste en temas oscuros. En esta ejecución, la página 1 del manual de instalación ocupa el primer lugar; la página 5 contiene los requisitos de instalación marcados como evidencia relevante. Si el RAG usa sólo `k=1`, el generador no recibe esa evidencia y una mala respuesta posterior sería esperable.

**Qué hacer:** antes de cambiar el prompt del LLM, prueba recuperar la página de instalación relevante en rango 1. Si la respuesta sigue fallando con ese contexto, entonces investiga la etapa de generación.

## 6. Comparar Recall@k y Precision@k al elegir `k`

El caso guiado explica una consulta; esta tabla promedia Recall@k y Precision@k para ver el efecto de cambiar `k` en todo el conjunto. Excluimos `R03` porque no tiene evidencia relevante por diseño. MRR ya se calculó como una métrica independiente en la sección anterior.

In [ ]:
aggregate_rows = []
for k in ks:
    aggregate_rows.append({
        'k': k,
        'mean_recall@k': sum(recall_at_k(ranking, k) for ranking in rankings) / len(rankings),
        'mean_precision@k': sum(precision_at_k(ranking, k) for ranking in rankings) / len(rankings),
    })

aggregate_metrics = pd.DataFrame(aggregate_rows)
print('Comparación agregada de Recall@k y Precision@k.')
display(aggregate_metrics.style.format({'mean_recall@k': '{:.3f}', 'mean_precision@k': '{:.3f}'}))



### Lectura del promedio: interpreta lo que produjo esta ejecución

Lee los valores de la tabla junto con los casos. Si recall aumenta al subir `k`, el contexto empieza a incluir evidencia que antes faltaba. Si precision baja, también está entrando más texto irrelevante. El MRR resume qué tan arriba apareció el primer resultado correcto; un valor bajo indica que el reordenamiento es una prioridad.

**Qué hacer:** no elijas `k` sólo por alcanzar recall alto. Primero intenta mejorar el ranking; después elige el menor `k` que preserve el recall requerido por la tarea y verifica las páginas recuperadas.

## 7. Preguntas trampa: cuando no debe existir evidencia

Una pregunta sobre pan de masa madre está fuera de los manuales de lavadora. Aquí el resultado deseable no es recuperar un fragmento parecido: es que el sistema reconozca la falta de soporte y se abstenga.

In [ ]:
trap_case = next(case for case in cases if case['category'] == 'trap')
trap_contexts = pd.DataFrame(
    sorted(trap_case['contexts'], key=lambda context: context['rank'])
)[['rank', 'id', 'relevant', 'source', 'page', 'text']]

display(trap_contexts)
print('Respuesta esperada:', trap_case['reference_answer'])
print('Respuesta del ejemplo:', trap_case['answer'])
print('¿Abstención correcta?:', trap_case['answer'] == trap_case['reference_answer'])
print('precision@3 para los fragmentos disponibles:', precision_at_k(trap_contexts.relevant.tolist(), 3))
print('recall@3 por convención:', recall_at_k(trap_contexts.relevant.tolist(), 3))


### Lectura de la pregunta trampa: abstención buena; `recall=1` no significa éxito

Todos los fragmentos son irrelevantes, por lo que `precision@3 = 0.000`. La implementación devuelve `recall@3 = 1.000` cuando no existe evidencia relevante: es una convención para evitar dividir entre cero, no una afirmación de que el sistema recuperó información útil. La señal real aquí es que la respuesta se abstiene.

**Qué hacer:** reportar cobertura/ranking sólo sobre preguntas respondibles y añadir una métrica de abstención para las trampas. Nunca promedies ese `recall=1` con los casos de evidencia real sin declararlo.

## 8. Diagnóstico por componente

La misma respuesta mala puede requerir cambios distintos según las señales del retriever.

In [ ]:
diagnostic_guide = pd.DataFrame([
    {'observed_signal': 'recall@k bajo', 'meaning': 'La evidencia necesaria no entra al contexto.', 'first_component_to_check': 'Corpus, chunking, embeddings o consulta.'},
    {'observed_signal': 'precision@k baja', 'meaning': 'Entra demasiado ruido al contexto.', 'first_component_to_check': 'Filtros, ranking, re-ranking o un k excesivo.'},
    {'observed_signal': 'MRR bajo con recall alto', 'meaning': 'La evidencia existe, pero llega tarde.', 'first_component_to_check': 'Ranking o re-ranking.'},
    {'observed_signal': 'Pregunta trampa', 'meaning': 'No hay evidencia dentro del corpus.', 'first_component_to_check': 'Abstención, umbral de recuperación y respuesta segura.'},
])
display(diagnostic_guide)

print('Diagnóstico de R02:', 'ranking/re-ranking' if guided_metrics['reciprocal_rank'] < 1 else 'sin problema de posición')


### Interpretación final del diagnóstico

`R02` tiene recall suficiente al incluir dos fragmentos, pero su MRR es menor que 1 y su precision baja al crecer `k`. Por ello el primer experimento debe atacar **ranking/re-ranking**, no cambiar el modelo generador. Esta separación evita optimizar el componente equivocado.

## Resumen y ejercicios

- `recall@k` responde: *¿trajimos la evidencia necesaria?*
- `precision@k` responde: *¿cuánto ruido entró al contexto?*
- MRR responde: *¿qué tan pronto apareció la primera evidencia útil?*
- Las preguntas sin evidencia se evalúan por abstención, no mezclándolas ingenuamente con recall.

1. Cambia `R02` para poner `C5` en rango 1 y predice qué valores cambian. Después ejecuta las celdas y compruébalo.
2. Añade un cuarto fragmento irrelevante a `R01`. ¿Qué ocurre con `precision@3`, `precision@4` y recall?
3. Elige un recall mínimo para un asistente de soporte y justifica qué `k` usarías antes y después de aplicar un re-ranker.